# 04 — Model Evaluation
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives

1. Compute **Precision, Recall, mAP50, mAP50-95** on the test set.
2. Visualise the **Confusion Matrix**.
3. Plot **Precision-Recall curves** per class.
4. Identify weak classes.
5. Save all figures to `outputs/images/`.

---

### Metrics Glossary

| Metric | Definition |
|--------|------------|
| **Precision** | Of all predicted positives, how many are correct? |
| **Recall** | Of all true positives, how many did we find? |
| **mAP50** | Mean Average Precision at IoU=0.50 |
| **mAP50-95** | Mean AP averaged over IoU thresholds 0.50→0.95 (stricter) |

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

_nb_dir = Path().resolve()
PROJECT_ROOT = _nb_dir
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from ultralytics import YOLO
from src.ppe_detection.utils import (
    MODELS_DIR, CONFIGS_DIR, EXPERIMENTS_DIR, OUTPUTS_DIR, CLASS_NAMES, ensure_dirs
)

ensure_dirs()

WEIGHTS  = MODELS_DIR / "yolov8n_ppe_baseline.pt"
DATA_YAML = CONFIGS_DIR / "ppe_dataset.yaml"
EXP_DIR  = EXPERIMENTS_DIR / "ppe_v1" / "baseline"

print(f"Weights: {WEIGHTS}")
print(f"Weights exist: {WEIGHTS.exists()}")

## 2. Run Validation on Test Split

In [ ]:
model = YOLO(str(WEIGHTS))
metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    verbose=True,
)
print("Validation complete.")

## 3. Overall Metrics

In [ ]:
results_summary = {
    "Precision":   round(float(metrics.box.mp), 4),
    "Recall":      round(float(metrics.box.mr), 4),
    "mAP50":       round(float(metrics.box.map50), 4),
    "mAP50-95":    round(float(metrics.box.map), 4),
}

summary_df = pd.DataFrame([results_summary])
print(summary_df.to_string(index=False))

## 4. Per-Class Metrics

In [ ]:
class_metrics = []
for i, name in enumerate(metrics.names.values()):
    class_metrics.append({
        "class": name,
        "precision": round(float(metrics.box.p[i]), 4),
        "recall":    round(float(metrics.box.r[i]), 4),
        "ap50":      round(float(metrics.box.ap50[i]), 4),
        "ap50_95":   round(float(metrics.box.ap[i]), 4),
    })

cls_df = pd.DataFrame(class_metrics)
print(cls_df.to_string(index=False))

# Save CSV
csv_path = OUTPUTS_DIR / "images" / "per_class_metrics.csv"
cls_df.to_csv(csv_path, index=False)
print(f"\nSaved → {csv_path}")

## 5. Confusion Matrix & PR Curves

Ultralytics saves these automatically during `model.val()`.  
We load and display them here.

In [ ]:
def show_and_save(src_path: Path, title: str, dest_path: Path) -> None:
    if not src_path.exists():
        print(f"Not found: {src_path}")
        return
    img = mpimg.imread(str(src_path))
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(str(dest_path), dpi=150)
    plt.show()
    print(f"Saved → {dest_path}")

val_dir = Path(metrics.save_dir)

show_and_save(
    val_dir / "confusion_matrix.png",
    "Confusion Matrix",
    OUTPUTS_DIR / "images" / "eval_confusion_matrix.png",
)
show_and_save(
    val_dir / "PR_curve.png",
    "Precision-Recall Curve",
    OUTPUTS_DIR / "images" / "eval_pr_curve.png",
)
show_and_save(
    val_dir / "F1_curve.png",
    "F1 Curve",
    OUTPUTS_DIR / "images" / "eval_f1_curve.png",
)

## 6. Analysis Checklist

After running the evaluation, answer these questions:

- [ ] Is `mAP50 > 0.70` overall?
- [ ] Which class has the **lowest** recall? Why?
- [ ] Are violation classes (`NO-Hardhat`, `NO-Safety Vest`) detected reliably?
- [ ] Does the confusion matrix show frequent misclassification between `Hardhat` ↔ `NO-Hardhat`?
- [ ] Is `Person` detected with high recall (> 0.85)?

Fill in observations in this cell after running.

## 7. Conclusions & Next Steps

**If results are satisfactory:** proceed to inference notebooks.  
**If mAP is too low:** consider:
- Upgrading to `yolov8s.pt` (more parameters).
- More epochs (150+).
- Data augmentation tuning in the YAML config.

**Next:** `05_image_inference.ipynb` — run the model on unseen images.